# Обзор базы продуктов (USDA + Open Food Facts)

Две таблицы продуктов в `data/processed/`:
- **`usda_foods.csv`** — базовые продукты (Foundation Foods), 469 наименований, полный набор нутриентов.
- **`off_ru_foods.csv`** — российские брендированные продукты из Open Food Facts.

В этом ноутбуке — обзор: размеры, шапки, заполненность, демо-поиск.

## 0. Загрузка

Ячейка читает файлы с диска ЗАНОВО при каждом запуске. Показывает дату изменения файла, чтобы было видно, насколько свежи данные.

In [1]:
import sys, os, datetime
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 40)

DATA = os.path.abspath('../data/processed')

usda = pd.read_csv(f'{DATA}/usda_foods.csv')
off_path = f'{DATA}/off_ru_foods.csv'

print(f'Папка данных: {DATA}')
print()
print(f'usda_foods.csv: {len(usda)} строк, изменён '
      f'{datetime.datetime.fromtimestamp(os.path.getmtime(f"{DATA}/usda_foods.csv")):%Y-%m-%d %H:%M}')

if os.path.exists(off_path):
    off = pd.read_csv(off_path)
    mtime = datetime.datetime.fromtimestamp(os.path.getmtime(off_path))
    print(f'off_ru_foods.csv: {len(off)} строк, изменён {mtime:%Y-%m-%d %H:%M}')
else:
    off = None
    print('off_ru_foods.csv: НЕ найден. Сначала запустите scripts/03_filter_off_ru.py')

Папка данных: z:\Projects\The_Diet_app\data\processed

usda_foods.csv: 469 строк, изменён 2026-07-15 21:53
off_ru_foods.csv: 31062 строк, изменён 2026-07-15 23:22


## 1. USDA — базовые продукты

In [2]:
usda.head()

,fdc_id,name,category,kcal,protein_g,fat_g,carbs_g,fiber_g,sugars_g,sat_fat_g,sodium_mg,potassium_mg,phosphorus_mg,calcium_mg,iron_mg,vitc_mg,thiamin_mg,riboflavin_mg,b12_ug
0,321358,"Hummus, commercial",Legumes and Legume Products,229.0,7.35,17.10,14.90,5.4,0.34,2.22,438.0,289.0,71.1,41.0,2.41,0.0,0.150,0.115,NaN
1,321359,"Milk, reduced fat, fluid, 2% milkfat...",Dairy and Egg Products,50.0,3.35,1.90,4.91,NaN,4.89,1.11,39.0,159.0,12.0,126.0,0.00,NaN,0.059,0.137,0.55
2,321360,"Tomatoes, grape, raw",Vegetables and Vegetable Products,27.0,0.83,0.63,5.51,2.1,NaN,NaN,6.0,260.0,11.9,11.0,0.33,27.2,0.075,0.065,NaN
3,321505,"Salt, table, iodized",Spices and Herbs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38700.0,2.0,0.0,50.0,0.00,NaN,NaN,NaN,NaN
4,321611,"Beans, snap, green, canned, regular ...",Vegetables and Vegetable Products,21.0,1.04,0.39,4.11,NaN,1.29,NaN,282.0,97.0,12.7,36.0,0.78,NaN,NaN,NaN,NaN


In [3]:
print('Заполненность нутриентов USDA (% непустых):')
(usda.drop(columns=['fdc_id']).notna().mean() * 100).round(0).to_frame('%')

Заполненность нутриентов USDA (% непустых):


,%
name,100.0
category,100.0
kcal,81.0
protein_g,91.0
fat_g,88.0
carbs_g,80.0
fiber_g,55.0
sugars_g,39.0
sat_fat_g,32.0
sodium_mg,86.0


In [4]:
print('Категории продуктов USDA:')
print(usda['category'].value_counts().to_string())

Категории продуктов USDA:
category
Vegetables and Vegetable Products    99
Fruits and Fruit Juices              75
Legumes and Legume Products          61
Dairy and Egg Products               50
Cereal Grains and Pasta              44
Finfish and Shellfish Products       24
Beef Products                        21
Nut and Seed Products                19
Sausages and Luncheon Meats          18
Fats and Oils                        12
Poultry Products                     12
Baked Products                        8
Pork Products                         7
Restaurant Foods                      5
Beverages                             4
Spices and Herbs                      3
Soups, Sauces, and Gravies            3
Sweets                                2
Lamb, Veal, and Game Products         2


## 2. OFF — российские брендированные продукты

In [5]:
if off is not None:
    off.head(10)
else:
    print('Файл off_ru_foods.csv не найден. Сначала запустите фильтрацию РФ.')

In [6]:
if off is not None:
    print('Заполненность нутриентов OFF-Россия (% непустых):')
    print((off.drop(columns=['code','brands','categories']).notna().mean() * 100).round(0).astype(int).astype(str) + '%')
    print()
    kbju = off[['kcal','protein_g','fat_g','carbs_g']].notna().all(axis=1)
    print(f'С полным КБЖУ: {kbju.sum()} продуктов ({kbju.sum()/len(off)*100:.0f}%)')

Заполненность нутриентов OFF-Россия (% непустых):
name            100%
kcal             40%
protein_g        41%
fat_g            41%
carbs_g          41%
fiber_g           3%
sugars_g          5%
sat_fat_g         3%
sodium_mg         4%
potassium_mg      0%
calcium_mg        0%
iron_mg           0%
vitc_mg           0%
dtype: str

С полным КБЖУ: 11376 продуктов (37%)


## 3. Демо-поиск продуктов

In [7]:
# Поиск по USDA — базовые продукты
query = 'apple'
usda[usda['name'].str.contains(query, case=False, na=False)][
    ['name','category','kcal','protein_g','fat_g','carbs_g','fiber_g']
].head(5)

,name,category,kcal,protein_g,fat_g,carbs_g,fiber_g
172,"Apples, red delicious, with skin, raw",Fruits and Fruit Juices,62.0,0.19,0.21,14.8,2.0
173,"Apples, honeycrisp, with skin, raw",Fruits and Fruit Juices,60.0,0.10,0.10,14.7,1.7
174,"Apples, granny smith, with skin, raw",Fruits and Fruit Juices,59.0,0.27,0.14,14.2,2.5
175,"Apples, gala, with skin, raw",Fruits and Fruit Juices,61.0,0.13,0.15,14.8,2.1
176,"Apples, fuji, with skin, raw",Fruits and Fruit Juices,65.0,0.15,0.16,15.7,2.1


In [8]:
# Поиск по OFF — российские бренды
# попробуйте: бородинский, простоквашино, савушкин, слобода, кефир, творог
if off is not None:
    query = 'кефир'
    found = off[off['name'].str.contains(query, case=False, na=False)]
    print(f'Найдено по «{query}»: {len(found)}')
    if len(found):
        found[['name','brands','kcal','protein_g','fat_g','carbs_g']].head(8)

Найдено по «кефир»: 164


## 4. Сводка

Две готовые таблицы — фундамент под следующую задачу (поиск продукта + расчёт рациона и привязка к нормам калькулятора).

In [9]:
summary = pd.DataFrame({
    'Источник': ['USDA Foundation Foods', 'Open Food Facts (Россия)'],
    'Продуктов': [len(usda), len(off) if off is not None else 0],
    'Тип': ['Базовые (яблоко, мясо, крупы)', 'РФ-бренды (йогурты, хлеб, сыры)'],
    'Файл': ['usda_foods.csv', 'off_ru_foods.csv'],
})
summary

,Источник,Продуктов,Тип,Файл
0,USDA Foundation Foods,469,"Базовые (яблоко, мясо, крупы)",usda_foods.csv
1,Open Food Facts (Россия),31062,"РФ-бренды (йогурты, хлеб, сыры)",off_ru_foods.csv
